In [0]:
# step 1--> read customers table from bronze layer and store in df
# step 2---> dispaly and identify the cleaning to be done

#step3---> apply required transformation steps which u identified and store in df 
#step4__> final df  we will apply scd  type 1 and store in silver schema  before silver schema using sql
#step5---> write a function in a same notebook clean and tranform the customer data from beonze and return df


#step 6---> write a function which can do scd1-if we provide teh parameters like source dataframe, business_key and target_table [after ctreating this function scd1 we will take it  to common_utils and reuse it here and in other notebooks


#step7--> Fully modularize the code ( hardcoded values to json config, proper logging setup and all the resusabl. 

# step 1--> read customers table from bronze layer and store in df


In [0]:
df= spark .read.table("retail_data_platform.bronze.sqlserver_customers")



# step 2---> display and identify the cleaning to be done


In [0]:
df.display()

#step3---> apply required transformation steps which u identified and store in df 

In [0]:
import pyspark.sql.functions as F

df= spark .read.table("retail_data_platform.bronze.sqlserver_customers")
#customer name should be small
df = df.withColumn("customer_name", F.lower(F.col("customer_name")))

#first name and last name should be seprated 
df = df.withColumn(
    "last_name",
    F.trim(F.split(F.col("customer_name"), ",").getItem(0))
).withColumn(
    "first_name",
    F.when(
        F.col("customer_name").contains(","),
        F.trim(F.split(F.col("customer_name"), ",").getItem(1))
    ).otherwise(None)
)


#drop customer
df = df.drop("customer_name")
#city should be capital
df = df.withColumn("city", F.upper(F.col("city")))

#.0 from postcode shoukld be removed 

df = df.withColumn(
    "postcode",
    F.regexp_replace(F.col("postcode").cast("string"), r"\.0$", "")
)
# drop filepath
df = df.drop("File_path")\
# add country coloumn with value USA
df = df.withColumn("country", F.lit("USA"))

#  valid from and valid to should be utc 
df = df.withColumn("valid_from", F.from_unixtime(F.expr("try_cast(valid_from as bigint)"))) \
        .withColumn("valid_to", F.from_unixtime(F.expr("try_cast(valid_to as bigint)")))
df.dropDuplicates(["customer_id"])
df.display()

#step4---> write a function in a same notebook clean and tranform the customer data from bronze and return d

In [0]:
import pyspark.sql.functions as F


def transform_customer_data(df):
    df = (
        df.withColumn("customer_name", F.lower(F.col("customer_name")))
        .withColumn(
            "last_name", F.trim(F.split(F.col("customer_name"), ",").getItem(0))
        )
        .withColumn(
            "first_name",
            F.when(
                F.col("customer_name").contains(","),
                F.trim(F.split(F.col("customer_name"), ",").getItem(1)),
            ).otherwise(None),
        )
        .withColumn("city", F.upper(F.col("city")))
        .withColumn(
            "postcode", F.regexp_replace(F.col("postcode").cast("string"), r"\.0$", "")
        )
        .withColumn("country", F.lit("USA"))
        .withColumn(
            "valid_from", F.from_unixtime(F.expr("try_cast(valid_from as bigint)"))
        )
        .withColumn("valid_to", F.from_unixtime(F.expr("try_cast(valid_to as bigint)")))
    )
    df = df.drop("customer_name", "file_path")
    df.dropDuplicates(["customer_id"])


    return df.select(
    "customer_id",
    "first_name",
    "last_name",
    "tax_id",
    "tax_code",
    "country",
    "state",
    "city",
    "postcode",
    "street",
    "number",
    "unit",
    "region",
    "district",
    "lon",
    "lat",
    "ship_to_address",
    "valid_from",
    "valid_to",
    "units_purchased",
    "loyalty_segment",
    "last_update_ts",
)

In [0]:
df= spark .read.table("retail_data_platform.bronze.sqlserver_customers")
cleaned_customer = transform_customer_data(df)
cleaned_customer.display()


#step5__> final df  we will apply scd  type 1 and store in silver schema  before silver schema using sql

In [0]:
if spark.catalog.tableExists("retail_data_platform.silver.customer"):
 print("table exist-proceding scd1")

else:
    print("Table Does Not Exist - Now Creating Table")
    spark.sql("""Create Schema if not exists retail_data_platform.silver""")
    cleaned_customer.write.format("delta").mode("overwrite").saveAsTable("retail_data_platform.silver.customers")
